In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

crime_df = spark.table("crime_parquet_table")

In [0]:
import time

def test_partitions(num_partitions):
    spark.conf.set("spark.sql.shuffle.partitions", num_partitions)
    
    start = time.time()
    crime_df.groupBy("District").count().collect()
    end = time.time()
    
    print(f"{num_partitions} partitions:", end - start)

test_partitions(200)
test_partitions(400)
test_partitions(800)

200 partitions: 0.7322189807891846
400 partitions: 0.4321413040161133
800 partitions: 0.5066463947296143


In [0]:
small_df = crime_df.sample(fraction=0.25, seed=42)
medium_df = crime_df.sample(fraction=0.5, seed=42)
large_df = crime_df

def test_dataset(df, label):
    spark.conf.set("spark.sql.shuffle.partitions", 400)
    
    start = time.time()
    df.groupBy("District").count().collect()
    end = time.time()
    
    print(f"{label}:", end - start)

test_dataset(small_df, "Small dataset")
test_dataset(medium_df, "Medium dataset")
test_dataset(large_df, "Large dataset")

Small dataset: 0.4908754825592041
Medium dataset: 0.7496933937072754
Large dataset: 0.6050193309783936


In [0]:
from pyspark.sql.functions import broadcast, when

district_lookup = crime_df.select("District").distinct()

district_lookup = district_lookup.withColumn(
    "District_Category",
    when(col("District") <= 10, "Low")
    .otherwise("High")
)

# Normal Join
start = time.time()
normal_join = crime_df.join(district_lookup, on="District", how="left")
normal_join.count()
end = time.time()

print("Normal Join Time:", end - start)

# Broadcast Join
start = time.time()
broadcast_join = crime_df.join(broadcast(district_lookup), on="District", how="left")
broadcast_join.count()
end = time.time()

print("Broadcast Join Time:", end - start)

Normal Join Time: 0.43398070335388184
Broadcast Join Time: 0.5267400741577148
